# Pegasus WMS: Real Estate Price Prediction Workflow

This notebook builds a complete **Pegasus Workflow Management System** pipeline for analyzing the Real Estate dataset and training multiple regression models to predict house prices.

## Workflow DAG
```
Real estate.csv
      |
      v
   EDA  ──► eda_report.txt, histogram.png, correlation.png
      |
      v
Preprocess  ──► train_x/y.csv, test_x/y.csv, scaler.pkl
      |
  ┌───┼───┐
  v   v   v
  LR  RF  GB   ──► model.pkl, metrics.json
  │   │   │
  └───┼───┘
      v
  Evaluate  ──► report.txt, best_model.pkl, predictions.csv
```

**Models trained:** Linear Regression, Random Forest, Gradient Boosting (trained in parallel)

In [ ]:
# ──────────────────────────────────────────────────────
# 1. Setup: Install dependencies (run once)
# ──────────────────────────────────────────────────────
import subprocess, sys, os, json, pickle, shutil

# Install ML libraries needed for wrapper scripts
libs = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn']
for lib in libs:
    try:
        __import__(lib.replace('-', '_'))
        print(f'  ✓ {lib} already installed')
    except ImportError:
        print(f'  Installing {lib}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])
        print(f'  ✓ {lib} installed')

# Verify Pegasus
from Pegasus.api import *
print(f'  ✓ Pegasus WMS API loaded')

In [ ]:
# ──────────────────────────────────────────────────────
# 2. Explore the Real Estate dataset
# ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
DATA_PATH = 'Real estate.csv'
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}\n')
df.head(10)

In [ ]:
# ──────────────────────────────────────────────────────
# 3. Summary statistics & quick EDA
# ──────────────────────────────────────────────────────
# Identify target column
target_col = [c for c in df.columns if 'price' in c.lower()][0]
feature_cols = [c for c in df.columns if c not in ['No', target_col]]

print(f'Target: {target_col}')
print(f'Features ({len(feature_cols)}): {feature_cols}\n')

# Summary stats
display(df[feature_cols + [target_col]].describe())

# Missing values
print(f'\nMissing values:\n{df.isnull().sum()}')

# Correlation with target
print(f'\nCorrelation with target:')
display(df[feature_cols + [target_col]].corr()[target_col].sort_values(ascending=False))

In [ ]:
# ──────────────────────────────────────────────────────
# 4. Visualize the data
# ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution of target
axes[0].hist(df[target_col], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('House Price (per unit area)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of House Prices')
axes[0].grid(True, alpha=0.3)

# House age vs Price
axes[1].scatter(df['X2 house age'], df[target_col], alpha=0.5, color='coral')
axes[1].set_xlabel('House Age (years)')
axes[1].set_ylabel('Price')
axes[1].set_title('House Age vs Price')
axes[1].grid(True, alpha=0.3)

# Distance to MRT vs Price
axes[2].scatter(df['X3 distance to the nearest MRT station'], df[target_col], alpha=0.5, color='green')
axes[2].set_xlabel('Distance to MRT (m)')
axes[2].set_ylabel('Price')
axes[2].set_title('Distance to MRT vs Price')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eda_quick.png', dpi=100, bbox_inches='tight')
plt.show()
print('Quick EDA saved to eda_quick.png')

In [ ]:
# ──────────────────────────────────────────────────────
# 5. Correlation matrix
# ──────────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
corr = df[feature_cols + [target_col]].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print('Correlation matrix saved to correlation_matrix.png')

---
## Pegasus WMS Workflow Definition

Now we define the full Pegasus workflow. This will:
1. Create **wrapper scripts** for each pipeline step
2. Define **Transformations** (executables)
3. Define the **Site Catalog** (execution environment)
4. Define the **Replica Catalog** (input data files)
5. Build the **DAG** with proper dependencies
6. Write all catalogs and the workflow YAML

In [ ]:
# ──────────────────────────────────────────────────────
# 6. Create wrapper scripts for workflow jobs
# ──────────────────────────────────────────────────────
BIN_DIR = 'workflow_bin'
os.makedirs(BIN_DIR, exist_ok=True)

# ── 6a. EDA script ──
eda_script = '''#!/usr/bin/env python3
import argparse, sys, os
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', required=True)
    parser.add_argument('--output_report', required=True)
    parser.add_argument('--output_histogram', required=True)
    parser.add_argument('--output_corr', required=True)
    args = parser.parse_args()
    
    import pandas as pd, matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt, seaborn as sns, numpy as np
    
    df = pd.read_csv(args.input)
    df.columns = df.columns.str.strip()
    target = [c for c in df.columns if 'price' in c.lower()][0]
    features = [c for c in df.columns if c not in ['No', target]]
    
    lines = ['='*70, 'EDA REPORT - Real Estate Dataset', '='*70]
    lines.append(f'Shape: {df.shape}')
    lines.append(f'Target: {target}')
    lines.append(f'Features: {features}')
    lines.append('\n--- Summary ---\n' + df.describe().to_string())
    lines.append('\n--- Missing ---\n' + df.isnull().sum().to_string())
    corr = df[features + [target]].corr()[target].sort_values(ascending=False)
    lines.append('\n--- Correlation with Target ---\n' + corr.to_string())
    with open(args.output_report, 'w') as f: f.write('\n'.join(lines))
    print(f'[EDA] Report: {args.output_report}')
    
    fig, ax = plt.subplots(figsize=(10,6))
    ax.hist(df[target], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_xlabel('Price'); ax.set_ylabel('Frequency')
    ax.set_title('Distribution of House Prices'); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(args.output_histogram, dpi=100); plt.close()
    print(f'[EDA] Histogram: {args.output_histogram}')
    
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(df[features + [target]].corr(), annot=True, cmap='coolwarm', center=0, square=True, linewidths=0.5, ax=ax)
    ax.set_title('Correlation Matrix')
    plt.tight_layout(); plt.savefig(args.output_corr, dpi=100); plt.close()
    print(f'[EDA] Corr Matrix: {args.output_corr}')
    print('[EDA] Complete!')
if __name__ == '__main__': main()
'''
with open(os.path.join(BIN_DIR, 'eda.py'), 'w') as f: f.write(eda_script)

# ── 6b. Preprocess script ──
preprocess_script = '''#!/usr/bin/env python3
import argparse, sys, os, pickle
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', required=True)
    parser.add_argument('--output_train_x', required=True)
    parser.add_argument('--output_train_y', required=True)
    parser.add_argument('--output_test_x', required=True)
    parser.add_argument('--output_test_y', required=True)
    parser.add_argument('--output_scaler', required=True)
    parser.add_argument('--test_size', type=float, default=0.2)
    parser.add_argument('--random_state', type=int, default=42)
    args = parser.parse_args()
    
    import pandas as pd, numpy as np
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    
    df = pd.read_csv(args.input); df.columns = df.columns.str.strip()
    target = [c for c in df.columns if 'price' in c.lower()][0]
    features = [c for c in df.columns if c not in ['No', target]]
    X, y = df[features].values, df[target].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=args.test_size, random_state=args.random_state)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)
    pd.DataFrame(X_tr_s, columns=features).to_csv(args.output_train_x, index=False)
    pd.Series(y_tr, name=target).to_csv(args.output_train_y, index=False)
    pd.DataFrame(X_te_s, columns=features).to_csv(args.output_test_x, index=False)
    pd.Series(y_te, name=target).to_csv(args.output_test_y, index=False)
    with open(args.output_scaler, 'wb') as f: pickle.dump(scaler, f)
    print(f'[Prep] Train: {len(y_tr)}, Test: {len(y_te)}')
    print('[Prep] Complete!')
if __name__ == '__main__': main()
'''
with open(os.path.join(BIN_DIR, 'preprocess.py'), 'w') as f: f.write(preprocess_script)

# ── 6c. Train script ──
train_script = '''#!/usr/bin/env python3
import argparse, sys, pickle, json
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--train_x', required=True)
    parser.add_argument('--train_y', required=True)
    parser.add_argument('--model_type', required=True, choices=['linear_regression','random_forest','gradient_boosting'])
    parser.add_argument('--output_model', required=True)
    parser.add_argument('--output_metrics', required=True)
    parser.add_argument('--random_state', type=int, default=42)
    args = parser.parse_args()
    
    import pandas as pd, numpy as np
    from sklearn.linear_model import LinearRegression
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    X = pd.read_csv(args.train_x).values; y = pd.read_csv(args.train_y).values.ravel()
    
    if args.model_type == 'linear_regression':
        model = LinearRegression()
    elif args.model_type == 'random_forest':
        model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=args.random_state, n_jobs=-1)
    else:
        model = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=args.random_state)
    
    model.fit(X, y); y_pred = model.predict(X)
    metrics = {'model_type': args.model_type,
               'train_rmse': round(float(np.sqrt(mean_squared_error(y, y_pred))),4),
               'train_mae': round(float(mean_absolute_error(y, y_pred)),4),
               'train_r2': round(float(r2_score(y, y_pred)),4)}
    with open(args.output_model, 'wb') as f: pickle.dump(model, f)
    with open(args.output_metrics, 'w') as f: json.dump(metrics, f, indent=2)
    print(f'[Train-{args.model_type}] R2: {metrics["train_r2"]}')
    print(f'[Train-{args.model_type}] Complete!')
if __name__ == '__main__': main()
'''
with open(os.path.join(BIN_DIR, 'train.py'), 'w') as f: f.write(train_script)

# ── 6d. Evaluate script ──
evaluate_script = '''#!/usr/bin/env python3
import argparse, sys, json, pickle, numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--test_x', required=True); parser.add_argument('--test_y', required=True)
    parser.add_argument('--model_lr', required=True); parser.add_argument('--model_rf', required=True); parser.add_argument('--model_gb', required=True)
    parser.add_argument('--metrics_lr', required=True); parser.add_argument('--metrics_rf', required=True); parser.add_argument('--metrics_gb', required=True)
    parser.add_argument('--output_report', required=True); parser.add_argument('--output_best_model', required=True); parser.add_argument('--output_predictions', required=True)
    args = parser.parse_args()
    
    X_test = pd.read_csv(args.test_x).values; y_test = pd.read_csv(args.test_y).values.ravel()
    models, train_metrics = {}, {}
    for name, mp, mpath in [('Linear Regression', args.model_lr, args.metrics_lr), ('Random Forest', args.model_rf, args.metrics_rf), ('Gradient Boosting', args.model_gb, args.metrics_gb)]:
        with open(mp, 'rb') as f: models[name] = pickle.load(f)
        with open(mpath) as f: train_metrics[name] = json.load(f)
    
    results = []
    for name, m in models.items():
        yp = m.predict(X_test)
        results.append({'model': name, 'test_rmse': round(float(np.sqrt(mean_squared_error(y_test, yp))),4), 'test_r2': round(float(r2_score(y_test, yp)),4), 'train_r2': train_metrics[name]['train_r2']})
    results.sort(key=lambda x: x['test_r2'], reverse=True)
    best = results[0]
    
    lines = ['='*70, 'MODEL EVALUATION REPORT', '='*70, f'Test samples: {len(y_test)}']
    lines.append('\n--- Comparison ---')
    lines.append(f"{'Model':<22} {'Test RMSE':<12} {'Test R2':<10} {'Train R2':<10}")
    lines.append('-'*54)
    for r in results: lines.append(f"{r['model']:<22} {r['test_rmse']:<12} {r['test_r2']:<10} {r['train_r2']:<10}")
    lines.append(f'\n>>> BEST MODEL: {best["model"]} (Test R2 = {best["test_r2"]})')
    
    bm = models[best['model']]
    if hasattr(bm, 'feature_importances_'):
        fnames = pd.read_csv(args.test_x).columns.tolist()
        imp = bm.feature_importances_
        lines.append('\n--- Feature Importance ---')
        for i in np.argsort(imp)[::-1]: lines.append(f'  {fnames[i]:35} {imp[i]:.4f}')
    elif hasattr(bm, 'coef_'):
        fnames = pd.read_csv(args.test_x).columns.tolist()
        lines.append('\n--- Coefficients ---')
        for n, c in zip(fnames, bm.coef_): lines.append(f'  {n:35} {c:.4f}')
    
    with open(args.output_report, 'w') as f: f.write('\n'.join(lines))
    with open(args.output_best_model, 'wb') as f: pickle.dump(bm, f)
    pd.DataFrame({'actual': y_test, 'predicted': np.round(bm.predict(X_test),4), 'residual': np.round(y_test - bm.predict(X_test),4)}).to_csv(args.output_predictions, index=False)
    print(f'[Eval] Report: {args.output_report}')
    print(f'[Eval] Best model: {args.output_best_model}')
    print(f'[Eval] Predictions: {args.output_predictions}')
    print('[Eval] Complete!')
if __name__ == '__main__': main()
'''
with open(os.path.join(BIN_DIR, 'evaluate.py'), 'w') as f: f.write(evaluate_script)

# Make them executable
for fn in ['eda.py', 'preprocess.py', 'train.py', 'evaluate.py']:
    os.chmod(os.path.join(BIN_DIR, fn), 0o755)

print(f'✓ Created {len(os.listdir(BIN_DIR))} wrapper scripts in {BIN_DIR}/')
print('  - eda.py: Exploratory data analysis')
print('  - preprocess.py: Train/test split + scaling')
print('  - train.py: Model training (LR/RF/GB)')
print('  - evaluate.py: Model evaluation & selection')

In [ ]:
# ──────────────────────────────────────────────────────
# 7. Define Pegasus Catalogs
# ──────────────────────────────────────────────────────
from Pegasus.api import *

WORKFLOW_NAME = 'realestate-prediction'
WORK_DIR = os.getcwd()

# ── 7a. Site Catalog ──
sc = SiteCatalog()
local_site = Site('local', arch=Arch.X86_64, os_type=OS.LINUX)
local_site.add_directories(
    Directory(Directory.SHARED_STORAGE, path=WORK_DIR)
        .add_file(File('realestate.csv'))
)
sc.add_sites(local_site)
sc.write()
print('✓ sites.yml written')

# ── 7b. Transformation Catalog ──
tc = TransformationCatalog()

# EDA
eda_trans = Transformation('eda', site='local',
    pfn=os.path.join(WORK_DIR, BIN_DIR, 'eda.py'),
    is_stageable=True)

# Preprocess
preprocess_trans = Transformation('preprocess', site='local',
    pfn=os.path.join(WORK_DIR, BIN_DIR, 'preprocess.py'),
    is_stageable=True)

# Train (3 variants, same script, different arguments)
train_lr_trans = Transformation('train_linear_regression', site='local',
    pfn=os.path.join(WORK_DIR, BIN_DIR, 'train.py'),
    is_stageable=True)

train_rf_trans = Transformation('train_random_forest', site='local',
    pfn=os.path.join(WORK_DIR, BIN_DIR, 'train.py'),
    is_stageable=True)

train_gb_trans = Transformation('train_gradient_boosting', site='local',
    pfn=os.path.join(WORK_DIR, BIN_DIR, 'train.py'),
    is_stageable=True)

# Evaluate
evaluate_trans = Transformation('evaluate', site='local',
    pfn=os.path.join(WORK_DIR, BIN_DIR, 'evaluate.py'),
    is_stageable=True)

tc.add_transformations(eda_trans, preprocess_trans,
    train_lr_trans, train_rf_trans, train_gb_trans,
    evaluate_trans)
tc.write()
print('✓ tc.yml written')

# ── 7c. Replica Catalog (input data) ──
rc = ReplicaCatalog()
input_file = File('realestate.csv')
rc.add_replica('local', 'realestate.csv', os.path.join(WORK_DIR, 'Real estate.csv'))
rc.write()
print('✓ rc.yml written')

In [ ]:
# ──────────────────────────────────────────────────────
# 8. Build the DAG with all jobs
# ──────────────────────────────────────────────────────
wf = Workflow(WORKFLOW_NAME, infer_dependencies=True)

# Define all logical files
eda_report      = File('eda_report.txt')
histogram_plot  = File('histogram.png')
correlation_plot = File('correlation_matrix.png')

train_x  = File('train_x.csv')
train_y  = File('train_y.csv')
test_x   = File('test_x.csv')
test_y   = File('test_y.csv')
scaler   = File('scaler.pkl')

lr_model    = File('lr_model.pkl')
lr_metrics  = File('lr_metrics.json')
rf_model    = File('rf_model.pkl')
rf_metrics  = File('rf_metrics.json')
gb_model    = File('gb_model.pkl')
gb_metrics  = File('gb_metrics.json')

eval_report    = File('evaluation_report.txt')
best_model     = File('best_model.pkl')
predictions    = File('predictions.csv')

# ── Job 1: EDA ──
j_eda = Job(eda_trans) \
    .add_inputs(input_file) \
    .add_outputs(eda_report, histogram_plot, correlation_plot) \
    .add_args('--input', input_file,
              '--output_report', eda_report,
              '--output_histogram', histogram_plot,
              '--output_corr', correlation_plot)

# ── Job 2: Preprocess ──
j_prep = Job(preprocess_trans) \
    .add_inputs(input_file) \
    .add_outputs(train_x, train_y, test_x, test_y, scaler) \
    .add_args('--input', input_file,
              '--output_train_x', train_x,
              '--output_train_y', train_y,
              '--output_test_x', test_x,
              '--output_test_y', test_y,
              '--output_scaler', scaler)

# ── Job 3a: Linear Regression ──
j_lr = Job(train_lr_trans) \
    .add_inputs(train_x, train_y) \
    .add_outputs(lr_model, lr_metrics) \
    .add_args('--train_x', train_x, '--train_y', train_y,
              '--model_type', 'linear_regression',
              '--output_model', lr_model,
              '--output_metrics', lr_metrics)

# ── Job 3b: Random Forest ──
j_rf = Job(train_rf_trans) \
    .add_inputs(train_x, train_y) \
    .add_outputs(rf_model, rf_metrics) \
    .add_args('--train_x', train_x, '--train_y', train_y,
              '--model_type', 'random_forest',
              '--output_model', rf_model,
              '--output_metrics', rf_metrics)

# ── Job 3c: Gradient Boosting ──
j_gb = Job(train_gb_trans) \
    .add_inputs(train_x, train_y) \
    .add_outputs(gb_model, gb_metrics) \
    .add_args('--train_x', train_x, '--train_y', train_y,
              '--model_type', 'gradient_boosting',
              '--output_model', gb_model,
              '--output_metrics', gb_metrics)

# ── Job 4: Evaluate (fan-in) ──
j_eval = Job(evaluate_trans) \
    .add_inputs(test_x, test_y,
                lr_model, rf_model, gb_model,
                lr_metrics, rf_metrics, gb_metrics) \
    .add_outputs(eval_report, best_model, predictions,
                stage_out=True) \
    .add_args('--test_x', test_x, '--test_y', test_y,
              '--model_lr', lr_model,
              '--model_rf', rf_model,
              '--model_gb', gb_model,
              '--metrics_lr', lr_metrics,
              '--metrics_rf', rf_metrics,
              '--metrics_gb', gb_metrics,
              '--output_report', eval_report,
              '--output_best_model', best_model,
              '--output_predictions', predictions)

# Add all jobs to the workflow
wf.add_jobs(j_eda, j_prep, j_lr, j_rf, j_gb, j_eval)

# Attach catalogs
wf.add_site_catalog(sc)
wf.add_transformation_catalog(tc)
wf.add_replica_catalog(rc)

# Write the workflow YAML
wf.write()
print('✓ workflow.yml written')

In [ ]:
# ──────────────────────────────────────────────────────
# 9. Visualize & verify the DAG
# ──────────────────────────────────────────────────────
print('=' * 60)
print(f'Workflow: {WORKFLOW_NAME}')
print(f'Jobs: {len(wf.jobs)}')
print(f'Dependencies: auto-inferred from File sharing')
print('=' * 60)
print()
print('DAG Structure:')
print('  realestate.csv')
print('       │')
print('       ├──► eda.py ──► eda_report.txt, histogram.png, correlation.png')
print('       │')
print('       ├──► preprocess.py ──► train_x/y.csv, test_x/y.csv, scaler.pkl')
print('       │')
print('       │    ┌──► train.py (LR) ──► lr_model.pkl, lr_metrics.json')
print('       ├────┼──► train.py (RF) ──► rf_model.pkl, rf_metrics.json')
print('       │    └──► train.py (GB) ──► gb_model.pkl, gb_metrics.json')
print('       │')
print('       └──► evaluate.py ──► evaluation_report.txt,')
print('                            best_model.pkl,')
print('                            predictions.csv')
print()
print('─' * 60)
print('Output files generated in this directory:')
for f in ['sites.yml', 'tc.yml', 'rc.yml', 'workflow.yml']:
    size = os.path.getsize(f)
    print(f'  • {f} ({size} bytes)')
print()
print('─' * 60)
print('Generated files in workflow_bin/:')
for f in sorted(os.listdir(BIN_DIR)):
    print(f'  • {BIN_DIR}/{f}')

In [ ]:
# ──────────────────────────────────────────────────────
# 10. Run the workflow (Option A: Direct execution)
# ──────────────────────────────────────────────────────
# You can run each step directly to verify everything works:

print('Running EDA directly...')
!python3 workflow_bin/eda.py --input "Real estate.csv" \
    --output_report eda_report.txt \
    --output_histogram histogram.png \
    --output_corr correlation_matrix.png

print('\nRunning Preprocess directly...')
!python3 workflow_bin/preprocess.py --input "Real estate.csv" \
    --output_train_x train_x.csv --output_train_y train_y.csv \
    --output_test_x test_x.csv --output_test_y test_y.csv \
    --output_scaler scaler.pkl

print('\nTraining models...')
for model_type in ['linear_regression', 'random_forest', 'gradient_boosting']:
    model_file = f'{model_type.split("_")[0]}_model.pkl'
    metrics_file = f'{model_type.split("_")[0]}_metrics.json'
    !python3 workflow_bin/train.py \
        --train_x train_x.csv --train_y train_y.csv \
        --model_type {model_type} \
        --output_model {model_file} --output_metrics {metrics_file}

print('\nEvaluating models...')
!python3 workflow_bin/evaluate.py \
    --test_x test_x.csv --test_y test_y.csv \
    --model_lr lr_model.pkl --model_rf rf_model.pkl --model_gb gb_model.pkl \
    --metrics_lr lr_metrics.json --metrics_rf rf_metrics.json --metrics_gb gb_metrics.json \
    --output_report evaluation_report.txt \
    --output_best_model best_model.pkl \
    --output_predictions predictions.csv

print('\n✓ All steps completed successfully!')

In [ ]:
# ──────────────────────────────────────────────────────
# 11. View results
# ──────────────────────────────────────────────────────
print('\n' + '='*60)
print('EDA REPORT')
print('='*60)
with open('eda_report.txt') as f:
    print(f.read())

print('\n' + '='*60)
print('EVALUATION REPORT')
print('='*60)
with open('evaluation_report.txt') as f:
    print(f.read())

print('\n' + '='*60)
print('PREDICTIONS (first 10 rows)')
print('='*60)
pred_df = pd.read_csv('predictions.csv')
print(pred_df.head(10).to_string())

In [ ]:
# ──────────────────────────────────────────────────────
# 12. (Optional) Plan & Submit via Pegasus
# ──────────────────────────────────────────────────────
# Uncomment to plan and submit the workflow through Pegasus:

# import subprocess
# 
# print('Planning workflow...')
# result = subprocess.run([
#     'pegasus-plan',
#     '--sites', 'local',
#     '--output-sites', 'local',
#     '--submit',
#     'workflow.yml'
# ], capture_output=True, text=True)
# print(result.stdout)
# if result.returncode != 0:
#     print('STDERR:', result.stderr)
# else:
#     print('Workflow submitted! Use pegasus-status to monitor.')

In [ ]:
# ──────────────────────────────────────────────────────
# 13. Cleanup (optional)
# ──────────────────────────────────────────────────────
# Remove generated files to start fresh:
# 
# import os, shutil
# for f in ['sites.yml', 'tc.yml', 'rc.yml', 'workflow.yml',
#           'eda_report.txt', 'histogram.png', 'correlation_matrix.png',
#           'train_x.csv', 'train_y.csv', 'test_x.csv', 'test_y.csv', 'scaler.pkl',
#           'lr_model.pkl', 'lr_metrics.json',
#           'rf_model.pkl', 'rf_metrics.json',
#           'gb_model.pkl', 'gb_metrics.json',
#           'evaluation_report.txt', 'best_model.pkl', 'predictions.csv',
#           'eda_quick.png']:
#     if os.path.exists(f): os.remove(f)
# shutil.rmtree('workflow_bin', ignore_errors=True)
# shutil.rmtree('output', ignore_errors=True)
# print('Cleanup complete')

---
## Summary

### Files Generated
| File | Purpose |
|------|---------|
| `workflow.yml` | Main Pegasus workflow DAG |
| `sites.yml` | Site catalog (local execution) |
| `tc.yml` | Transformation catalog (6 jobs) |
| `rc.yml` | Replica catalog (input data) |
| `workflow_bin/eda.py` | Exploratory data analysis |
| `workflow_bin/preprocess.py` | Data preprocessing |
| `workflow_bin/train.py` | Model training (3 variants) |
| `workflow_bin/evaluate.py` | Model evaluation & selection |

### To submit via Pegasus
```bash
pegasus-plan --sites local --output-sites local --submit workflow.yml
```

### To monitor
```bash
pegasus-status <submit-dir>
pegasus-analyzer <submit-dir>
```